In [0]:
%pip install geopandas rasterio shapely

In [0]:
import os
import glob
import tempfile
import shutil
import traceback
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from rasterio.enums import Resampling
from shapely.geometry import mapping

In [0]:
flight_list = dbutils.jobs.taskValues.get(
    taskKey="orquestator", 
    key="missing_clips", 
    default=[]
)

if not flight_list:
    dbutils.notebook.exit("No pending flights to process. Exiting gracefully.")

print(f" Received {len(flight_list)} flights for plot cropping.\n")

for flight_path in flight_list:
    print("-" * 60)
    print(f" PROCESSING FLIGHT: {flight_path}")
    
    if flight_path.startswith("dbfs:/"):
        flight_path = flight_path.replace("dbfs:/", "/dbfs/")
        
    base_dir = os.path.dirname(flight_path)
    parent_dir = os.path.dirname(base_dir)
    field_data_dir = os.path.join(parent_dir, "field_data")

    possible_ortho_names = ["RGB.tif", "MS.tif"]
    rgb_path = None

    for name in possible_ortho_names:
        candidate = f"{base_dir}/*/*/*/{name}"
        matches = glob.glob(candidate)
        if matches:
            rgb_path = matches[0]
            break

    print(f" Searching geometries in: {field_data_dir}")
    vector_path = os.path.join(field_data_dir, "location_boundary.geojson")
    
    if rgb_path is None:
        print(f" Error: No RGB.tif or MS.tif found in {base_dir}. Skipping flight...")
        continue
    else:
        ortho_name = os.path.splitext(os.path.basename(rgb_path))[0]
        print(f" Orthomosaic found: {os.path.basename(rgb_path)}")
        # Los recortes se guardarán en la misma carpeta que el .tif original
        out_dir = os.path.dirname(rgb_path)
        
    if not os.path.exists(vector_path):
        print(f" Error: No 'location_boundary.geojson' found in {field_data_dir}. Skipping flight...") 
        continue 
        
    print(f" Vector file found: {os.path.basename(vector_path)}")

    os.makedirs(out_dir, exist_ok=True)

    # Local scratch directory on the cluster's own disk
    local_tmp_dir = tempfile.mkdtemp(prefix="plot_clip_")

    try:
        print(" Loading geometries and checking Coordinate Reference Systems (CRS)...")
        gdf_plots = gpd.read_file(vector_path)
        
        with rasterio.open(rgb_path) as src:
            raster_crs = src.crs
            src_colorinterp = src.colorinterp
            
            if gdf_plots.crs != raster_crs:
                print(f" Reprojecting polygons from {gdf_plots.crs} to {raster_crs}...")
                gdf_plots = gdf_plots.to_crs(raster_crs)

            n_bands = src.count
            print(f" Detected {n_bands} bands.")

            # =====================================================================
            # RECORTAR Y GUARDAR EL .TIF CON EL BOUNDARY COMPLETO
            # =====================================================================
            print(" Clipping the original .tif file based on the full boundary...")
            all_geometries = [mapping(geom) for geom in gdf_plots.geometry]
            out_image_full, out_transform_full = mask(src, all_geometries, crop=True)
            
            out_meta_full = src.meta.copy()
            out_meta_full.update({
                "driver": "GTiff",
                "height": out_image_full.shape[1],
                "width": out_image_full.shape[2],
                "transform": out_transform_full,
                "compress": "lzw",  # Recommended compression for TIFs
                "BIGTIFF": "YES"
            })
            
            # Se guarda con el sufijo _clipped para no corromper el archivo que se está leyendo
            clipped_tif_name = f"{ortho_name}_clipped.tif"
            local_full_tif_path = os.path.join(local_tmp_dir, clipped_tif_name)
            
            with rasterio.open(local_full_tif_path, "w", **out_meta_full) as dest:
                dest.write(out_image_full)
                dest.colorinterp = src_colorinterp
                
            final_full_tif_path = os.path.join(out_dir, clipped_tif_name)
            shutil.copyfile(local_full_tif_path, final_full_tif_path)
            
            print(f" SUCCESS: Clipped TIF saved to {final_full_tif_path}")

    except Exception as e:
        print(f" An error occurred processing this flight: {e}")
        traceback.print_exc()
    
    finally:
        # Clean up local scratch files regardless of success/failure
        shutil.rmtree(local_tmp_dir, ignore_errors=True)

print("\n" + "="*70)
print(" LOCATION CROPPING PIPELINE FINISHED SUCCESSFULLY.")

In [ ]:
# ======================================================================
# STANDALONE JOB: generate PNG previews for the CLIPPED orthomosaics
# produced by the cropping cell ({ortho}_clipped.tif).
#
# Robust validity mask: uses src.read_masks() so it works regardless of
# how the TIF encodes transparency (alpha band, nodata, internal mask).
#
# Requires rasterio and Pillow on the cluster:
#   %pip install rasterio pillow
# ======================================================================
import os, shutil
import numpy as np
import rasterio
from rasterio.enums import Resampling, ColorInterp
from PIL import Image
import pyspark.sql.functions as F

# =============================== PARAMETERS ============================
CATALOG = "apse2_prod_irri_fg_catalog_7474658213144266"  # change
SCHEMA  = "tier1_raw"                                     # change

ortho_table = f"{CATALOG}.{SCHEMA}.drone_ortho_table"

ORTHO_PATH_COLUMN = "ortho_file_path"   # column holding the ORIGINAL .tif path
CLIPPED_SUFFIX    = "_clipped"          # suffix added by the cropping cell
FORCE_REGENERATE  = False               # True to rebuild PNGs even if they exist

# ------------------------------ PNG config ----------------------------
MAX_DIM     = 4096        # max PNG size (px). Set to None for full resolution.
STRETCH_PCT = (2, 98)     # contrast stretch by percentiles
# For multispectral: which spectral bands map to R, G, B (1-based).
# DJI P4M / MicaSense order is usually Blue, Green, Red, RedEdge, NIR,
# so natural color = (3, 2, 1). Change this if your camera differs.
MS_RGB_BANDS = (3, 2, 1)
# ======================================================================


# --------------------------- PNG generation ---------------------------
def get_spectral_bands(src):
    """Return (spectral_bands_1based, alpha_index_or_None)."""
    spectral, alpha = [], None
    for i, ci in enumerate(src.colorinterp, start=1):
        if ci == ColorInterp.alpha:
            alpha = i
        else:
            spectral.append(i)
    return spectral, alpha


def stretch_band(band, low, high):
    """Rescale a single band to 8-bit using the given low/high bounds."""
    if high <= low:
        return np.zeros(band.shape, dtype="uint8")
    out = np.clip((band.astype("float32") - low) / (high - low), 0, 1) * 255.0
    return out.astype("uint8")


def tif_to_png(src_tif, dst_png, sensor_type):
    with rasterio.open(src_tif) as src:
        # Downscale factor so the longest side fits within MAX_DIM.
        scale = min(1.0, MAX_DIM / max(src.width, src.height)) if MAX_DIM else 1.0
        out_h = max(1, int(round(src.height * scale)))
        out_w = max(1, int(round(src.width * scale)))

        spectral, alpha_idx = get_spectral_bands(src)

        # Pick the R, G, B bands.
        if sensor_type == "MS" and len(spectral) >= 3:
            chosen = [spectral[b - 1] for b in MS_RGB_BANDS]
        elif len(spectral) >= 3:
            chosen = spectral[:3]
        else:                       # single band -> replicate into grayscale
            chosen = [spectral[0]] * 3

        data = src.read(chosen, out_shape=(3, out_h, out_w),
                        resampling=Resampling.bilinear).astype("float32")

        # -----------------------------------------------------------------
        # Robust validity mask via read_masks(): rasterio/GDAL resolves it
        # from the alpha band, the nodata value, or an internal .msk mask,
        # whichever the file uses. A pixel is valid only where ALL chosen
        # spectral bands are valid. Falls back to nodata/finite checks.
        # -----------------------------------------------------------------
        try:
            band_masks = src.read_masks(chosen, out_shape=(3, out_h, out_w),
                                        resampling=Resampling.nearest)
            valid = np.all(band_masks > 0, axis=0)
        except Exception:
            # Extremely defensive fallback (older rasterio / odd drivers).
            if alpha_idx is not None:
                valid = src.read(alpha_idx, out_shape=(out_h, out_w),
                                 resampling=Resampling.nearest) > 0
            elif src.nodata is not None:
                valid = np.all(data != src.nodata, axis=0)
            else:
                valid = np.ones((out_h, out_w), dtype=bool)

        # Discard non-finite pixels (NaN/inf) regardless of the mask source.
        valid &= np.all(np.isfinite(data), axis=0)

        # Per-band contrast stretch using only valid pixels.
        rgb = np.zeros((out_h, out_w, 3), dtype="uint8")
        for k in range(3):
            vals = data[k][valid]
            if vals.size:
                low, high = np.percentile(vals, STRETCH_PCT)
                rgb[..., k] = stretch_band(data[k], low, high)

        # RGBA output: empty/nodata areas become transparent.
        alpha_ch = np.where(valid, 255, 0).astype("uint8")
        Image.fromarray(np.dstack([rgb, alpha_ch]), mode="RGBA").save(dst_png)


def normalize_path(p):
    """Make a stored path usable by os/shutil (dbfs:/ -> /dbfs/)."""
    return p.replace("dbfs:/", "/dbfs/") if p else p


def clipped_tif_path(orig_tif):
    """RGB.tif -> RGB_clipped.tif (same folder, produced by the cropping cell)."""
    base, ext = os.path.splitext(orig_tif)
    return f"{base}{CLIPPED_SUFFIX}{ext}"


def detect_sensor_type(tif_path):
    """Infer MS vs RGB from the filename (MS... -> MS, otherwise RGB)."""
    return "MS" if os.path.basename(tif_path).upper().startswith("MS") else "RGB"


# --------------------- Select orthos that exist -----------------------
raw_ortho_df = spark.table(ortho_table)

# Same rule as your inventory: an ortho is "finished" when ortho_exists is True.
if "ortho_exists" in raw_ortho_df.columns:
    finish_df = raw_ortho_df.filter(F.col("ortho_exists") == True)
else:
    finish_df = raw_ortho_df

# Pull the ORIGINAL .tif paths directly from the ortho table.
ortho_rows = (finish_df
              .select(ORTHO_PATH_COLUMN)
              .where(F.col(ORTHO_PATH_COLUMN).isNotNull())
              .distinct()
              .collect())

ortho_paths = [r[ORTHO_PATH_COLUMN] for r in ortho_rows]

print("-" * 40)
print(f"Orthomosaics to inspect: {len(ortho_paths)}")
print("-" * 40)


# --------------------------- Main loop --------------------------------
generated, skipped, errors, missing_clip = 0, 0, 0, 0

for raw_path in ortho_paths:
    orig_tif = normalize_path(raw_path)

    if not orig_tif.lower().endswith((".tif", ".tiff")):
        print(f"--   Skipping non-tif path: {orig_tif}")
        continue

    # We work on the CLIPPED ortho produced by the cropping cell, not the original.
    tif = clipped_tif_path(orig_tif)

    if not os.path.exists(tif):
        missing_clip += 1
        print(f"--   Clipped ortho not found (crop pending?): {tif}")
        continue

    sensor_type = detect_sensor_type(tif)                 # RGB_clipped -> RGB, MS_clipped -> MS
    png_path    = os.path.splitext(tif)[0] + ".png"       # RGB_clipped.png (same base as the clipped tif)

    # THE CHECK: only generate when the PNG is missing (unless forced).
    if os.path.exists(png_path) and not FORCE_REGENERATE:
        skipped += 1
        print(f"SKIP {png_path}  (already exists)")
        continue

    # Work on local disk to avoid Volume I/O errors.
    tmp_tif = "/tmp/_ortho_src.tif"
    tmp_png = "/tmp/_ortho_out.png"
    try:
        shutil.copy2(tif, tmp_tif)
        tif_to_png(tmp_tif, tmp_png, sensor_type)
        shutil.copy2(tmp_png, png_path)
        generated += 1
        print(f"OK   {tif}  ->  {png_path}")
    except Exception as e:
        errors += 1
        print(f"ERR  {tif}: {e}")
    finally:
        for f in (tmp_tif, tmp_png):
            if os.path.exists(f):
                os.remove(f)

# ------------------------------ Summary -------------------------------
summary = (f"Done. PNG generated: {generated} | "
           f"Skipped (already had PNG): {skipped} | "
           f"Clipped ortho missing: {missing_clip} | Errors: {errors}")
print("\n" + summary)
dbutils.notebook.exit(summary)